# 05a Registration gate: FPFH + RANSAC + ICP (+ scale-aware and finer ICP, issue #20) on 9 PC2Model samples

**Question.** Does the classical registration from notebook 02 (voxel downsample, FPFH features, RANSAC, point-to-plane
ICP), followed by the refinement tuned in issue #20 (below), recover PC2Model's ground-truth transform on 9 samples: 3 houses, 3 simulated indoor scenes and 3 real indoor scans?
This is the registration half of the roadmap's Phase 0.5 gate. The other halves (perturbed model, hand check) are 05b and later.

**Pre-tuning result, kept as context.** This notebook was first run with 02's pipeline unchanged (RANSAC + coarse ICP, scale from
02's RMS-radius estimate): **4 of 9** samples succeeded (needed 6). Rotation was within 2 degrees on all nine; the four
scaled samples all failed the 1% scale criterion (the estimate was 1.5%, 3.9%, 5.3% and 7.4% off); only 5 of 9 centroids
were within 5 cm, because ICP ran only at the coarse voxel (0.18 to 0.44 m).

**Tuning (issue #20).** A refinement after the coarse ICP was added: scale-fitting ICP stages (so the scale comes from
correspondences, not the RMS radius) and finer ICP stages, in `scan_vs_bim.register`. Its parameters were chosen in
notebook `05c_registration_tuning.ipynb` on held-out PC2Model samples (`fur_010`, `house_6`, `indoor_modelling_5-GM`,
`real-indoor_modelling_5-GM`, and a synthetically scaled GM), none of them one of the nine here or of the same building, and
**not on this notebook's results**. This notebook was then run once with the choice: same nine samples, same seeds, same
thresholds, same pick rule (the highest *coarse* ICP fitness among the five seeds; the refinement runs on the picked draw).

**Slice.** Per category (house, `indoor_modelling` = simulated indoor, `ISPRS_indoor_modeling_benchmark` = real indoor):
the smallest sample with a ground-truth scale different from 1, the smallest sample with scale 1, then the next smallest
remaining one (all by compressed size, so the downloads stay small). `scripts/pick_gate_samples.py` made the choice
before this notebook ran and wrote `outputs/05a/gate_samples.csv`.

**Thresholds, fixed before any result.** Verbatim from `docs/roadmap.md`. They are judgement calls, not taken from a standard.

- Registration success for one sample, after ICP: rotation error <= 2 degrees, translation error <= 5 cm, and scale error
  <= 1% on scaled samples. *(Since issue #38 and `docs/adr/0001-fit-scale-on-every-sample.md` the scale criterion applies to
  every sample, not only the scaled ones: the registration fits scale on every sample, so an unscaled sample that comes out
  at a scale of 1.03 fails. The quote above is the roadmap's wording, kept as written.)*
- The registration part of the gate passes if at least 6 of the 9 samples succeed.
- Perturbed CRAS model (a later notebook): every injected change beyond tolerance gets the right verdict, and at most 10%
  of untouched elements are falsely flagged shifted or missing.
- Hand check of 20 random real CRAS elements (a later notebook): at least 80% agree with the verdict.

Only the first two bullets are tested here; the other two belong to later notebooks.

**Decisions made before running** (also judgement calls).

1. *Translation error* is the **centroid error**: the distance between where the estimate and the ground truth put the
   scan's centroid. The difference of the two translation vectors, `|dt|`, is shown for context only, because a small
   rotation error times the scan's distance from its own origin (hundreds of metres here) dominates it.
2. *A sample counts on the draw picked by ICP fitness* among seeds 0 to 4 (the highest fitness; the ground truth is not
   used to pick). Notebook 02 showed that RANSAC is a lottery on a real room and that the seed declared in advance can be a
   failure, so one seed per sample would mostly measure luck. Seed 0's result is shown too, as context only.
3. *Scale* (issue #38, ADR 0001). On every sample the scan is first multiplied by 02's RMS-radius estimate, then the
   pipeline runs (RANSAC, coarse ICP, and since #20 the refinement, whose scale-fitting stages correct that estimate), and
   the transforms are composed (`T = T_refined * diag(s, s, s, 1)`). The benchmark's scaled / unscaled flag (a sample is
   "scaled" when its ground-truth scale is not 1, `|s - 1| > 0.001`) is no longer an input: it is kept as a reporting column
   (the `scaled` column of `registration.csv`, the per-group counts). The scale criterion applies to every sample.
4. The voxel fraction (0.008 of the model's bounding-box diagonal), the five seeds, the feature radii and the RANSAC and ICP
   settings are 02's, unchanged. They were tuned on UVigo's ground truth in 02, which is one of the nine samples here. The
   refinement stages (`scan_vs_bim.register.refine_plan`) are new in #20 and were chosen on 05c's held-out samples.

In [1]:
import csv
import gc
import io
import logging
import time
from pathlib import Path

import numpy as np
import open3d as o3d
from remotezip import RemoteZip

from scan_vs_bim.fetch import extract_zip_member, http_range_reader
from scan_vs_bim.load import load_e57
from scan_vs_bim.memory import peak_rss_gb
from scan_vs_bim.register import CENTROID_MAX_M, ROT_MAX_DEG, SCALE_MAX_REL, SEEDS, apply, register_sample, score, seed_all
from scan_vs_bim.pc2model import URL as PC2MODEL_URL, Sample, parse_transform, pick_gate_samples, scale_of

o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "pc2model"
OUT = ROOT / "outputs" / "05a"
OUT.mkdir(parents=True, exist_ok=True)
t_notebook = time.perf_counter()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# ---- Every knob of this notebook is in this cell, except those of scan_vs_bim.register (seeds, features, RANSAC, ICP, thresholds). ---------------------------------------------------------------------------------

# Registration success thresholds (ROT_MAX_DEG, CENTROID_MAX_M, SCALE_MAX_REL: docs/roadmap.md, judgement calls, not from a standard)
# are in scan_vs_bim.register; see the first cell.
GATE_MIN_SUCCESS = 6   # of the 9 samples

# A sample is "scaled" when its ground-truth scale differs from 1 by more than this.
SCALED_TOL = 1e-3

# Seed for the model-surface sampling and the scan subsample. The RANSAC draws (one per seed in register.SEEDS, reseeded
# before every RANSAC run) are in scan_vs_bim.register; the draw with the highest ICP fitness counts.
SEED = 0

# Point budgets (02's values).
MODEL_POINTS = 500_000        # points sampled uniformly from the model mesh surface
MAX_SCAN_POINTS = 1_000_000   # at most this many scan points are loaded, read in chunks (see the memory note below)
CHUNK_POINTS = 1_000_000      # E57 points read per chunk

# Voxel size = VOXEL_FRACTION * diagonal of the model's bounding box (02's value).
VOXEL_FRACTION = 0.008

# Direction check on the ground truth: T must put the scan within this fraction of the model's diagonal (median distance),
# and inv(T) must be at least INV_FACTOR times worse. This is a direction check, not an accuracy check: it separates T from
# inv(T), which is off by orders of magnitude. The 2% limit is loose on purpose (house_18 passes at 0.406 m against a ~0.52 m
# limit because its scan covers far more than its model, so even the true T leaves a 0.406 m median residual).
DIRECTION_FRACTION, INV_FACTOR = 0.02, 10

## The nine samples

The notebook reads `outputs/05a/gate_samples.csv` (written by `scripts/pick_gate_samples.py` from the zip's central
directory and the 137 transform files; no point cloud is downloaded for the choice). As a check, the choice is recomputed
here from `outputs/pc2model_scales.csv` and compared.

In [3]:
gate_csv, scales_csv = OUT / "gate_samples.csv", ROOT / "outputs" / "pc2model_scales.csv"
for p, how in [(gate_csv, "scripts/pick_gate_samples.py"), (scales_csv, "scripts/find_scaled_sample.py")]:
    if not p.exists():
        raise FileNotFoundError(f"{p.relative_to(ROOT)} is missing. Run: uv run python {how}")
chosen = list(csv.DictReader(open(gate_csv)))
print(f"{'category':18s}{'sample':72s}{'MB':>8s}{'scale':>8s}")
for r in chosen:
    print(f"{r['label']:18s}{Path(r['sample']).name:72s}{r['compressed_mb']:>8s}{float(r['scale']):8.3f}")

# Consistency check, not a known answer: re-deriving the selection rule from the scales CSV (sizes rounded to 0.01 MB there)
# must reproduce the choice stored in gate_samples.csv. Both come from the same rule, so this catches drift, not a wrong rule.
rows = list(csv.DictReader(open(scales_csv)))
by_folder = {r["sample"]: float(r["scale"]) for r in rows}
all_samples = [Sample(r["sample"], round(float(r["compressed_mb"]) * 1e6), None) for r in rows]
categories = list(dict.fromkeys(r["folder_category"] for r in chosen))
assert [s.folder for s in pick_gate_samples(all_samples, by_folder, categories)] == [r["sample"] for r in chosen]
assert len(chosen) == 9 and all(abs(float(r["scale"]) - by_folder[r["sample"]]) < 1e-6 for r in chosen)
print(f"\n{sum(abs(float(r['scale']) - 1) > SCALED_TOL for r in chosen)} of the 9 are scaled (ground-truth scale != 1).")

category          sample                                                                        MB   scale
house             house_23-sketchfab                                                        157.63   1.206
house             house_13-sketchfab                                                        103.40   1.000
house             house_18-sketchfab                                                        152.80   1.000
simulated indoor  indoor_modelling_4-ISPRS_indoor_modelling-UoM                             151.65   0.747
simulated indoor  indoor_modelling_3-ISPRS_indoor_modelling-UVigo                           141.40   1.000
simulated indoor  indoor_modelling_0-ISPRS_indoor_modelling-TUBS1                           189.19   0.689
real indoor       real-indoor_modelling_1-ISPRS_indoor_modelling-TUBS2                      225.44   1.258
real indoor       real-indoor_modelling_3-ISPRS_indoor_modelling-UVigo                      102.09   1.000
real indoor       real-indoor_modelli

## The method, stage by stage

In `scan_vs_bim.register` (`register_sample` and `score`, moved out of this notebook in #37; the numbers are unchanged). For each sample, one at a time:

1. **Fetch** the sample's three files (model OBJ, scan E57, ground-truth transform) by HTTP range requests if they are
   not in `data/pc2model/` yet. Resumable, CRC-checked (`scan_vs_bim.fetch`). Nothing else of the 10 GB zip is read.
2. **Load.** Sample the OBJ surface uniformly into a point cloud; load the scan with a seeded subsample.
3. **Scale** (every sample, ADR 0001): `s = RMS radius of the model / RMS radius of the scan`, each about its own centroid, on
   the voxel-downsampled clouds. The scan is multiplied by `s`. This is only the start: the refinement (step 5) fits the scale.
4. **Voxel size** = 0.008 x the model's bounding-box diagonal. Normals and FPFH on the downsampled clouds, RANSAC over FPFH
   correspondences, then point-to-plane ICP. Five RANSAC seeds; the draw with the highest ICP fitness counts.
5. **Refine** (#20, #38): `refine_icp(.., **refine_plan(voxel))` on the picked draw (and on seed 0, context only), on every sample:
   two point-to-point ICP stages that also fit a uniform scale (voxel/2, voxel/4), then point-to-plane rigid stages
   (voxel/4, voxel/8, ... down to 2.5 cm).
6. **Metrics against the ground truth `T`:** rotation error (geodesic angle, scale divided out), centroid error (the
   gate's "translation error"), `|dt|` for context, scale error, and the registration residual (median distance from
   every loaded scan point, moved by the estimate, to the model mesh; ray casting).
7. **Free** everything before the next sample.

**Memory.** On this 8 GB machine a loader that reads one whole scan before subsampling needs about 100 bytes per
point at its peak (1.08 GB on UVigo's single scan of 9.5M points). Most samples here are 8 to 12 scans of at most 1.7M points,
but the real indoor TUBS2 sample is one scan of 21.7M points, which would need about 2.2 GB. `load_e57` therefore reads
1M points at a time (`CHUNK_POINTS`) and picks the same points whatever the chunk size (`tests/test_load.py`); its peak on UVigo
was 0.30 GB, and it does not grow with the scan.

In [4]:
_INFOS = None   # the zip's central directory, fetched once and only if a sample has to be downloaded


def ensure_files(folder):
    # Returns (directory, retry lines printed by the downloader or None when nothing had to be fetched).
    global _INFOS
    d = DATA / Path(folder).name
    if d.exists() and all(any(d.glob(g)) for g in ("model_*.obj", "pointcloud_*_transformed.e57", "transformation_matrix_*.txt")):
        return d, None
    if _INFOS is None:
        with RemoteZip(PC2MODEL_URL) as z:
            _INFOS = z.infolist()
    reader, log = http_range_reader(PC2MODEL_URL), io.StringIO()
    handler, fetch_log = logging.StreamHandler(log), logging.getLogger("scan_vs_bim.fetch")
    fetch_log.addHandler(handler)
    fetch_log.setLevel(logging.INFO)
    fetch_log.propagate = False   # captured only, as the redirected stdout was
    try:
        for info in _INFOS:
            if info.filename.startswith(folder + "/") and not info.is_dir():
                extract_zip_member(reader, info, d / Path(info.filename).name)
    finally:
        fetch_log.removeHandler(handler)
        fetch_log.propagate = True
    return d, sum("retry" in line for line in log.getvalue().splitlines())


def load_sample(folder):
    d, retries = ensure_files(folder)
    mesh = o3d.io.read_triangle_mesh(str(next(d.glob("model_*.obj"))))
    T_gt = parse_transform(next(d.glob("transformation_matrix_*.txt")).read_text())
    seed_all(SEED)
    model = mesh.sample_points_uniformly(MODEL_POINTS)
    scan = load_e57(next(d.glob("pointcloud_*_transformed.e57")), MAX_SCAN_POINTS, SEED, CHUNK_POINTS)
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh))
    diag = float(np.linalg.norm(mesh.get_axis_aligned_bounding_box().get_extent()))
    return dict(model=model, scan=scan, T_gt=T_gt, scene=scene, diag=diag, n_triangles=len(mesh.triangles), retries=retries)


def median_to_model(scene, T, pts):
    return float(np.median(scene.compute_distance(o3d.core.Tensor(apply(T, pts).astype(np.float32))).numpy()))


In [5]:
def run_sample(r):
    t0 = time.perf_counter()
    s = load_sample(r["sample"])
    t_load = time.perf_counter() - t0
    pts, T_gt, diag = np.asarray(s["scan"].points), s["T_gt"], s["diag"]
    gt_scale = scale_of(T_gt)
    scaled = abs(gt_scale - 1) > SCALED_TOL
    voxel = VOXEL_FRACTION * diag

    # Direction check: T maps the scan onto the model, inv(T) does not.
    d_T, d_inv = median_to_model(s["scene"], T_gt, pts), median_to_model(s["scene"], np.linalg.inv(T_gt), pts)
    assert d_T < DIRECTION_FRACTION * diag and d_inv > INV_FACTOR * d_T, (r["sample"], d_T, d_inv, diag)
    print(f"  direction check {Path(r['sample']).name}: median distance under T {d_T:.3f} m, under inv(T) {d_inv:.1f} m "
          f"(limit for T {DIRECTION_FRACTION * diag:.3f} m)", flush=True)

    # 05a's registration, in the package: the RMS scale estimate on every sample (no use of the benchmark's flag, ADR 0001; `scaled` is
    # reporting only), a coarse draw per seed, the draw with the highest coarse ICP fitness (ties: the earlier seed), refinement
    # (issue #20: scale stages, then rigid ones) of it and of seed 0.
    res = register_sample(s["scan"], s["model"], voxel)
    t_reg, t_ref = res.coarse_s, res.refine_s
    best = next(d for d in res.draws if d.seed == res.seed)
    sc, sc0, sc_coarse = score(res.T, T_gt, pts), score(res.T_seed0, T_gt, pts), score(best.T, T_gt, pts)
    row = dict(
        sample=Path(r["sample"]).name, category=r["label"], folder=r["sample"], compressed_mb=float(r["compressed_mb"]),
        scaled=scaled, gt_scale=gt_scale, est_scale=scale_of(res.T), scale_err_pct=100 * sc.scale_err,
        rot_deg=sc.rot_deg, centroid_m=sc.centroid_m, dt_m=float(np.linalg.norm(res.T[:3, 3] - T_gt[:3, 3])),
        resid_median_m=median_to_model(s["scene"], res.T, pts), gt_resid_median_m=d_T,
        icp_fitness=best.fitness, best_seed=res.seed, success=sc.success,
        seed0_rot_deg=sc0.rot_deg, seed0_centroid_m=sc0.centroid_m, seed0_success=sc0.success,
        seeds_rot_deg=";".join(f"{score(d.T, T_gt, pts).rot_deg:.3f}" for d in res.draws),
        seeds_icp_fitness=";".join(f"{d.fitness:.3f}" for d in res.draws),
        n_scan_points=len(pts), model_diag_m=diag, voxel_m=voxel, n_triangles=s["n_triangles"],
        load_s=t_load, register_s=t_reg,
        coarse_rot_deg=sc_coarse.rot_deg, coarse_centroid_m=sc_coarse.centroid_m,
        coarse_scale_err_pct=100 * sc_coarse.scale_err, refine_s=t_ref)
    note = "" if s["retries"] is None else f", downloaded ({s['retries']} retries)"
    print(f"{row['sample']}: {row['n_triangles']:,} triangles, diagonal {diag:.1f}, {len(pts):,} scan points, voxel {voxel:.3f}; "
          f"load {t_load:.0f} s{note}, register {t_reg:.0f} s, refine {t_ref:.0f} s; "
          f"seed {row['best_seed']} picked, {'success' if row['success'] else 'fail'}; peak RSS so far {peak_rss_gb():.2f} GB", flush=True)
    return row

## Running the nine samples

One at a time: fetch, load, register, record, free. Each finished sample is appended to the list, and
`outputs/05a/registration.csv` (read by 05b) is written when the loop ends.

In [6]:
results = []
for r in chosen:
    results.append(run_sample(r))
    gc.collect()   # drop the clouds, mesh and ray-casting scene of this sample before the next one

with open(OUT / "registration.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(results[0]))
    w.writeheader()
    w.writerows(results)
print(f"\nwrote outputs/05a/registration.csv ({len(results)} rows)")

  direction check house_23-sketchfab: median distance under T 0.001 m, under inv(T) 63.7 m (limit for T 0.448 m)


house_23-sketchfab: 116,681 triangles, diagonal 22.4, 999,998 scan points, voxel 0.179; load 8 s, register 234 s, refine 85 s; seed 1 picked, success; peak RSS so far 0.24 GB


  direction check house_13-sketchfab: median distance under T 0.001 m, under inv(T) 136.8 m (limit for T 0.523 m)


house_13-sketchfab: 6,348 triangles, diagonal 26.1, 1,000,001 scan points, voxel 0.209; load 6 s, register 439 s, refine 136 s; seed 0 picked, success; peak RSS so far 0.24 GB


  direction check house_18-sketchfab: median distance under T 0.406 m, under inv(T) 184.1 m (limit for T 0.518 m)


house_18-sketchfab: 4,035 triangles, diagonal 25.9, 1,000,001 scan points, voxel 0.207; load 7 s, register 179 s, refine 547 s; seed 4 picked, success; peak RSS so far 0.24 GB


... (9 outputs left out)

real-indoor_modelling_3-ISPRS_indoor_modelling-UVigo: 79,736 triangles, diagonal 49.3, 1,000,000 scan points, voxel 0.394; load 7 s, register 81 s, refine 564 s; seed 0 picked, fail; peak RSS so far 0.24 GB


  direction check real-indoor_modelling_2-ISPRS_indoor_modelling-Fire_Brigade: median distance under T 0.049 m, under inv(T) 28.7 m (limit for T 1.102 m)


real-indoor_modelling_2-ISPRS_indoor_modelling-Fire_Brigade: 115,578 triangles, diagonal 55.1, 1,000,000 scan points, voxel 0.441; load 21 s, register 93 s, refine 646 s; seed 1 picked, success; peak RSS so far 0.25 GB



wrote outputs/05a/registration.csv (9 rows)


## Result: the counted draw (highest ICP fitness of 5 seeds)

Columns: ground-truth scale, rotation error, translation error (the **centroid error**; `|dt|` is context only), the
registration residual (median scan-to-model distance after registration, and the same under the ground truth `T`, the `resid at T` column, which is the best any registration can do on that sample: where the scan covers more than the model it is not near 0),
and whether the sample succeeded under the gate (rotation <= 2 deg, centroid <= 5 cm, and, on every sample since #38 / ADR 0001,
scale error <= 1%). The scale error column is the fitted scale against the truth, on every row, scaled or not.

In [7]:
def table(rows, seed0=False):
    rot, cen, ok = ("seed0_rot_deg", "seed0_centroid_m", "seed0_success") if seed0 else ("rot_deg", "centroid_m", "success")
    head = f"{'sample':62s}{'category':18s}{'GT':>7s}{'rot err':>9s}{'centroid':>10s}"
    unit = f"{'':62s}{'':18s}{'scale':>7s}{'(deg)':>9s}{'err (m)':>10s}"
    if not seed0:
        head += f"{'|dt|':>9s}{'resid med':>11s}{'resid at T':>11s}{'scale err':>10s}"
        unit += f"{'(m)':>9s}{'(m)':>11s}{'(m)':>11s}{'(%)':>10s}"
    print(head + "  result\n" + unit)
    for m in rows:
        line = f"{m['sample'].replace('ISPRS_indoor_modelling-', ''):62s}{m['category']:18s}{m['gt_scale']:7.3f}{m[rot]:9.2f}{m[cen]:10.3f}"
        if not seed0:
            line += f"{m['dt_m']:9.2f}{m['resid_median_m']:11.3f}{m['gt_resid_median_m']:11.3f}" + f"{m['scale_err_pct']:10.2f}"
        print(line + f"  {'ok' if m[ok] else 'FAIL'}")


table(results)

sample                                                        category               GT  rot err  centroid     |dt|  resid med resid at T scale err  result
                                                                                  scale    (deg)   err (m)      (m)        (m)        (m)       (%)
house_23-sketchfab                                            house               1.206     0.00     0.000     0.02      0.001      0.001      0.02  ok
house_13-sketchfab                                            house               1.000     0.01     0.001     0.01      0.001      0.001      0.01  ok
house_18-sketchfab                                            house               1.000     0.03     0.001     0.10      0.406      0.406      0.09  ok
indoor_modelling_4-UoM                                        simulated indoor    0.747     0.00     0.000     0.00      0.001      0.001      0.01  ok
indoor_modelling_3-UVigo                                      simulated indoor    1.000 

In [8]:
n_ok = sum(m["success"] for m in results)
n_seed0 = sum(m["seed0_success"] for m in results)
by_cat = {c: sum(m["success"] for m in results if m["category"] == c) for c in dict.fromkeys(m["category"] for m in results)}
by_scaled = {k: (sum(m["success"] for m in results if m["scaled"] == k), sum(m["scaled"] == k for m in results)) for k in (False, True)}
print(f"Succeeded (best of {len(SEEDS)} seeds by ICP fitness): {n_ok} of {len(results)}")
print("  per category: " + ", ".join(f"{c} {n}/3" for c, n in by_cat.items()))
print(f"  scale 1: {by_scaled[False][0]}/{by_scaled[False][1]}, scaled: {by_scaled[True][0]}/{by_scaled[True][1]}")
print(f"  context, seed 0 alone: {n_seed0} of {len(results)}")
n_no_scale = sum(m["rot_deg"] <= ROT_MAX_DEG and m["centroid_m"] <= CENTROID_MAX_M for m in results)
print(f"  context, ignoring the scale criterion: {n_no_scale} of {len(results)}")
print(f"\nRegistration criterion (a judgement call): at least {GATE_MIN_SUCCESS} of {len(results)} succeed -> "
      f"{'PASS' if n_ok >= GATE_MIN_SUCCESS else 'FAIL'}")

# Independent check: re-read the CSV the next notebook will read and recompute success per row from its numeric columns and
# the threshold constants (not from the in-memory flags), then compare with the stored column and the count printed above.
saved = list(csv.DictReader(open(OUT / "registration.csv")))
recomputed = [float(r["rot_deg"]) <= ROT_MAX_DEG and float(r["centroid_m"]) <= CENTROID_MAX_M
              and float(r["scale_err_pct"]) <= 100 * SCALE_MAX_REL for r in saved]
assert len(saved) == 9 and recomputed == [r["success"] == "True" for r in saved] and sum(recomputed) == n_ok

Succeeded (best of 5 seeds by ICP fitness): 6 of 9
  per category: house 3/3, simulated indoor 1/3, real indoor 2/3
  scale 1: 3/5, scaled: 3/4
  context, seed 0 alone: 3 of 9
  context, ignoring the scale criterion: 6 of 9

Registration criterion (a judgement call): at least 6 of 9 succeed -> PASS


### Seed 0 alone (context only)

The same table for the seed declared in advance, without the ICP-fitness pick (also refined). Where it differs from the table above, the
pick mattered (or did harm, if seed 0 was the better draw).

In [9]:
table(results, seed0=True)

print(f"\nRotation error (deg) of each of the five coarse draws (before refinement), and the ICP fitness that picked one (* = picked):")
for m in results:
    errs, fits = m["seeds_rot_deg"].split(";"), m["seeds_icp_fitness"].split(";")
    cells_ = [f"{float(e):7.2f}/{float(f):.3f}{'*' if i == m['best_seed'] else ' '}" for i, (e, f) in enumerate(zip(errs, fits))]
    print(f"{m['sample'].replace('ISPRS_indoor_modelling-', ''):62s}" + " ".join(cells_))

sample                                                        category               GT  rot err  centroid  result
                                                                                  scale    (deg)   err (m)
house_23-sketchfab                                            house               1.206     0.00     0.000  ok
house_13-sketchfab                                            house               1.000     0.01     0.001  ok
house_18-sketchfab                                            house               1.000    13.75     1.066  FAIL
indoor_modelling_4-UoM                                        simulated indoor    0.747   160.43     6.927  FAIL
indoor_modelling_3-UVigo                                      simulated indoor    1.000     0.34     0.568  FAIL
indoor_modelling_0-TUBS1                                      simulated indoor    0.689     0.29     0.197  FAIL
real-indoor_modelling_1-TUBS2                                 real indoor         1.258     0.28     0.0

## Run summary

In [10]:
print(f"notebook wall time {time.perf_counter() - t_notebook:.0f} s; in-kernel peak RSS {peak_rss_gb():.2f} GB")
print(f"{'sample':62s}{'load (s)':>10s}{'register (s)':>14s}{'refine (s)':>12s}")
for m in results:
    print(f"{m['sample'].replace('ISPRS_indoor_modelling-', ''):62s}{m['load_s']:10.0f}{m['register_s']:14.0f}{m['refine_s']:12.0f}")

notebook wall time 5674 s; in-kernel peak RSS 0.25 GB
sample                                                          load (s)  register (s)  refine (s)
house_23-sketchfab                                                     8           234          85
house_13-sketchfab                                                     6           439         136
house_18-sketchfab                                                     7           179         547
indoor_modelling_4-UoM                                                18           199         991
indoor_modelling_3-UVigo                                              28           117         395
indoor_modelling_0-TUBS1                                              28            96         166
real-indoor_modelling_1-TUBS2                                         12           196         180
real-indoor_modelling_3-UVigo                                          7            81         564
real-indoor_modelling_2-Fire_Brigade                   

## Answer

Numbers below are from this notebook's run after issue #38 (ADR 0001): the benchmark's scaled flag is no longer used, so every
sample starts from 02's RMS-radius scale estimate, gets the scale-fitting stages, and must have scale error <= 1%. Everything
else is the #20 run's (same nine samples, seeds, refinement plan chosen in 05c, pick rule). The non-refinement parameters are
02's, chosen after seeing UVigo's ground truth (one of the nine samples), and each sample is one sample.

**Does the registration criterion pass?** Yes, on the line: **6 of 9** samples succeed, against the 6 the roadmap asks for
(with the flag, the #20 run: 7 of 9; before the refinement: 4 of 9). By category: house 3 of 3, simulated indoor 1 of 3, real
indoor 2 of 3. Scale 1: 3 of 5; scaled: 3 of 4. Seed 0 alone would give 3 of 9. No margin is left: one more failure fails it.

**What #38 changed** (rotation deg / centroid m / scale error %, picked draw; `start` = the RMS estimate's scale error, 0 with
the flag on an unscaled sample). The four scaled samples run the same computation as before and their rows are identical to the
#20 run's, so only the five unscaled samples are listed:

| sample (scale 1) | with the flag (#20 run) | start | no flag (#38) |
|---|---|---|---|
| house_13 | 0.01 / 0.001 / 0.00, ok | 2.6 % | 0.01 / 0.001 / 0.01, ok |
| house_18 | 0.01 / 0.001 / 0.00, ok | 7.7 % | 0.03 / 0.001 / 0.09, ok |
| indoor_3 UVigo | 0.00 / 0.002 / 0.00, ok | 12.5 % | 0.34 / 0.568 / 6.04, **fail** |
| real_3 UVigo | 0.23 / 0.123 / 0.00, fail | 12.6 % | 0.30 / 1.479 / 8.31, **fail** |
| real_2 Fire Brigade | 0.04 / 0.002 / 0.00, ok | 0.1 % | 0.05 / 0.010 / 0.08, ok |

**Does fitting scale hurt samples that are not scaled?** Yes, measurably (ADR 0001 asked for this to be measured, not assumed).
Three of the five unscaled samples come through: the scale stages take house_13's 2.6 % and house_18's 7.7 % start errors to
0.01 % and 0.09 %, and Fire Brigade's estimate is already right. The two UVigo samples do not: the RMS-radius estimate is 12.5 %
off on both (probably because the scan and the model spread differently about their centroids, which the ratio of RMS radii
reads as scale; not checked), the
coarse pose is then fitted at the wrong scale (centroid 1.5 and 2.2 m off), and the scale stages, whose capture range in 05c was
2.2-3.6 %, end at 6.0 % and 8.3 %. Simulated indoor UVigo turns from a success at 2 mm into a failure at 57 cm; real UVigo was
already a failure (12.3 cm) and gets worse (1.48 m). The residual under the estimate on simulated UVigo is 1.3 cm against 0.1 cm
under the ground truth, so it is a wrong local fit, not a better one. house_18 shows that a 7.7 % start can be recovered, so the
capture range is wider than 05c tested on some samples, but not on these. ADR 0001 says what to do if this happens: revisit with
something like a user-declared "scan is metric" switch or a better scale start, not by bringing the oracle flag back. That is a
follow-up, not part of #38.

**Cost of the #38 run.** Registration (five seeds) 81 to 439 s per sample, refinement 85 to 991 s, whole notebook 95 min, peak RSS
0.25 GB, on a heavily shared machine (load average 40-85, other sessions running notebooks); these times are not comparable with the
#20 run's 25 min.

### The #20 run, with the flag (history)

The refinement (#20) was first run with the flag: 7 of 9 succeeded (house 3 of 3, simulated indoor 2 of 3, real indoor 2 of 3;
scale 1: 4 of 5; scaled: 3 of 4, before: 0 of 4; seed 0 alone 4 of 9, before: 2).

**Before the refinement, then after** (rotation deg / centroid m / scale error %, picked draw; the picks are unchanged because
the pick rule still uses the coarse ICP fitness):

| sample | before | after |
|---|---|---|
| house_23 (scaled 1.21) | 0.23 / 0.070 / 3.90, fail | 0.00 / 0.000 / 0.02, ok |
| house_13 | 0.02 / 0.002, ok | 0.01 / 0.001, ok |
| house_18 | 0.54 / 0.041, ok | 0.01 / 0.001, ok |
| indoor_4 UoM (scaled 0.75) | 0.05 / 0.024 / 1.55, fail | 0.00 / 0.000 / 0.01, ok |
| indoor_3 UVigo | 0.09 / 0.023, ok | 0.00 / 0.002, ok |
| indoor_0 TUBS1 (scaled 0.69) | 0.13 / 0.171 / 5.29, fail | 0.29 / 0.197 / 2.04, **fail** |
| real_1 TUBS2 (scaled 1.26) | 0.18 / 0.221 / 7.40, fail | 0.28 / 0.026 / 0.20, ok |
| real_3 UVigo | 0.33 / 0.139, fail | 0.23 / 0.123, **fail** |
| real_2 Fire Brigade | 0.20 / 0.033, ok | 0.04 / 0.002, ok |

**What changed.** The scale stages took three of the four scaled failures to within 0.2% (RMS-estimate errors of 3.9, 1.5 and
7.4% corrected to 0.02, 0.01 and 0.20%), and the finer stages brought the centroid error of seven of the nine samples to 2.6 cm
or less, mostly a millimetre or two. Rotation stays within 2 degrees on all nine (largest 0.29).

**Where it still fails.**

- *`indoor_modelling_0` (TUBS1, scaled 0.69).* The RMS estimate was 5.3% off; the scale stages moved it to 2.04% off but not
  to 1%, and the centroid stays at 19.7 cm. The residual to the model is 2.2 cm against 0.1 cm under the ground truth, so the
  refinement seems to have stopped in a local minimum, not at the truth. This start error is beyond what 05c tested (2.2 to 3.6%).
- *`real-indoor_modelling_3` (UVigo, scale 1).* Centroid error 12.3 cm (before: 13.9 cm) with rotation 0.23 degrees. The
  residual under the estimate (3.0 cm) is lower than under the ground truth (4.2 cm), so ICP fits this scan to the model better
  than the benchmark's own transform does; a finer ICP cannot reach the ground truth here. It may be the limit of the
  centroid criterion on real scans with clutter, not of the method. This sample is also the one 02 was tuned on.
- *Picking by ICP fitness* still chose the right rotation on all nine (as before; two samples, house_18 and Fire Brigade,
  succeed only because of the pick: their seed 0 is a coarse failure, shown in the seed 0 table).

**Cost.** Registration (five seeds) 13 to 146 s per sample; refinement 8 to 508 s (house_18 took 508 s, probably because its scan covers far more than its model, so the ICP
stages run to their iteration limits; not checked; the rest took under 170 s). Whole notebook 25 min on a shared
machine, peak RSS 0.43 GB.

**Limits.** One run of each sample (RANSAC seeds are fixed, Open3D is multithreaded; the pre-tuning run reproduced to about 1e-4
degrees on a second run, this one was not repeated); the scan is a 1M-point subsample; the scale handling then used the
benchmark's scaled / unscaled flag, which a real run does not have (#38 dropped it, above); the thresholds are judgement calls, not from a standard;
the refinement was tuned on four usable samples (05c). The scale stages cannot recover from a large start error (05c's real
GM sample, 26.7% off, failed), so a scaled scan whose RMS estimate is far off would still fail.